In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../")))

# data
from src.data.load_data import load_raw_data
from src.features.build_features import create_features

# models
from src.models.sarimax import SARIMAXModel
from src.models.ml_models import XGBoostModel, RandomForestModel

# walk-forward
from src.backtests.walk_forward import walk_forward_validation

# metrics
from src.evaluation.metrics import regression_metrics

In [ ]:
df = load_raw_data()
df = create_features(df)
df = df.sort_index().dropna()

print(df.shape)
df.head()

In [ ]:
wf_sarimax = walk_forward_validation(
    df,
    SARIMAXModel,
    window=300,
    step=5
)

wf_sarimax.head()

In [ ]:
wf_xgb = walk_forward_validation(
    df,
    XGBoostModel,
    window=300,
    step=5
)

wf_xgb.head()

In [ ]:
def add_price_columns(df):
    df = df.copy()
    df["Pred_Price"] = np.exp(df["Pred"])
    df["Actual_Price"] = np.exp(df["Actual"])
    return df

wf_sarimax = add_price_columns(wf_sarimax)
wf_xgb = add_price_columns(wf_xgb)

In [ ]:
plt.figure(figsize=(12,5))

plt.plot(wf_sarimax["Date"], wf_sarimax["Actual_Price"], label="Actual")
plt.plot(wf_sarimax["Date"], wf_sarimax["Pred_Price"], label="SARIMAX")

plt.title("Walk-Forward: SARIMAX")
plt.legend()
plt.grid()
plt.show()

In [ ]:
plt.figure(figsize=(12,5))

plt.plot(wf_xgb["Date"], wf_xgb["Actual_Price"], label="Actual")
plt.plot(wf_xgb["Date"], wf_xgb["Pred_Price"], label="XGBoost")

plt.title("Walk-Forward: XGBoost")
plt.legend()
plt.grid()
plt.show()

In [ ]:
def evaluate_wf(res):

    metrics = regression_metrics(
        res["Actual_Price"],
        res["Pred_Price"],
        is_log=False
    )

    direction = (res["Pred"] > 0) == (res["Actual"] > 0)

    return {
        "RMSE": metrics["RMSE"],
        "MAE": metrics["MAE"],
        "Direction_Accuracy": direction.mean()
    }

In [ ]:
sarimax_perf = evaluate_wf(wf_sarimax)
xgb_perf = evaluate_wf(wf_xgb)

print("SARIMAX:", sarimax_perf)
print("XGBoost:", xgb_perf)

In [ ]:
comparison = pd.DataFrame([
    {"Model": "SARIMAX", **sarimax_perf},
    {"Model": "XGBoost", **xgb_perf}
])

comparison

In [ ]:
comparison.set_index("Model")[["RMSE", "Direction_Accuracy"]].plot(
    kind="bar",
    figsize=(8,4),
    title="Model Comparison (Walk-Forward)"
)

plt.grid()
plt.show()

In [ ]:
def simple_strategy(res):

    res = res.copy()

    res["signal"] = (res["Pred"] > 0).astype(int)
    res["strategy"] = res["signal"] * res["Actual"]

    res["cumulative"] = np.exp(res["strategy"].cumsum())

    return res

sarimax_bt = simple_strategy(wf_sarimax)
xgb_bt = simple_strategy(wf_xgb)

In [ ]:
plt.figure(figsize=(12,5))

plt.plot(sarimax_bt["Date"], sarimax_bt["cumulative"], label="SARIMAX")
plt.plot(xgb_bt["Date"], xgb_bt["cumulative"], label="XGBoost")

plt.title("Strategy Performance")
plt.legend()
plt.grid()
plt.show()